# 12 — Translation: reading a document in the page's other language

A document keeps its own language: its key points, overview and map reasons are written in it. A student who
switches the page to Vietnamese while reading an English document saw English. This notebook picks a model that
translates those texts **on this machine** (no AI requests; the free Gemini tier allows about 20 a day), and
checks how well it does on the app's own texts, not only on a benchmark.

**Data.** IWSLT'15 English–Vietnamese (Luong & Manning 2015): TED talk subtitles, 133,318 training pairs and
1,268 test pairs (tst2013). HuggingFace parquet copy `IWSLT/mt_eng_vietnamese`, kept out of git under
`datasets/raw/IWSLT/`. The app's own sample is 30 texts from the demo account (12 key point names, 10 key
points, 6 map reasons, 2 overviews) in `ml_pipeline/reports/translation_domain_texts.json`.

**Candidates.** `Helsinki-NLP/opus-mt-en-vi` (Marian, 72M parameters) and `VietAI/envit5-translation` (T5, 275M,
both directions in one model). Scores are sacreBLEU and chrF against the human subtitles.

In [1]:
import sys, html, json, re, time
from pathlib import Path
from collections import Counter
sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from ml_pipeline.notebook_setup import bootstrap
bootstrap()
import pandas as pd, sacrebleu, torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from ml_pipeline import config
from ml_pipeline.translator import split_sentences

RAW = config.REPO_ROOT / 'datasets' / 'raw' / 'IWSLT'
def pairs(split):
    df = pd.read_parquet(RAW / f'{split}.parquet')
    return [(html.unescape(t['en']), html.unescape(t['vi'])) for t in df['translation']]
train, test = pairs('train'), pairs('test')
print(f'{len(train)} training pairs, {len(test)} test pairs')

133318 training pairs, 1269 test pairs


## 1. Can the corpus choose the interface's words?

The first idea was to let the corpus say how people translate the interface's words (the complaint was "chưa
vững" shown as "Shaky"). For each word, the Vietnamese 1–3 syllable phrases that appear with it most often,
scored by Dice (2·together / (alone₁ + alone₂)); the count in brackets is how often they appear together.

In [2]:
tok = lambda s: re.findall(r'[^\W\d_]+', s.lower())
EN = [tok(e) for e, _ in train]; VI = [tok(v) for _, v in train]
def grams(ws, n=3): return {' '.join(ws[i:i+k]) for k in range(1, n+1) for i in range(len(ws)-k+1)}
def has(ws, phrase):
    p = phrase.split(); return any(ws[i:i+len(p)] == p for i in range(len(ws)-len(p)+1))
def probe(phrase, src, tgt, top=5):
    hit = [i for i, ws in enumerate(src) if has(ws, phrase)]
    co = Counter(g for i in hit for g in grams(tgt[i]))
    cand = {g for g, c in co.items() if c >= 3}
    tot = Counter(g for ws in tgt for g in grams(ws) if g in cand)
    best = sorted(cand, key=lambda g: 2*co[g]/(len(hit)+tot[g]), reverse=True)[:top]
    return len(hit), ', '.join(f'{g} ({co[g]})' for g in best)
rows = [(w, *probe(w, EN, VI)) for w in ['shaky', 'solid', 'quiz', 'notebook', 'key points', 'practice', 'mistake']]
rows += [(w, *probe(w, VI, EN)) for w in ['chưa vững', 'nắm chắc', 'bài kiểm tra', 'luyện tập']]
pd.DataFrame(rows, columns=['phrase', 'sentences', 'most often with']).style.hide(axis='index')

phrase,sentences,most often with
shaky,8,"run (3), bị (3), cũng (3), rất (3), việc (3)"
solid,78,"rắn (22), chất rắn (5), rắn chắc (4), cứng (11), thể rắn (3)"
quiz,16,"quiz show (3), quiz (3), một câu đố (4), đố (7), câu đố (5)"
notebook,16,"cuốn sổ (4), sổ của (3), một quyển (3), sổ (8), vở (4)"
key points,0,
practice,173,"thực hành (26), luyện tập (23), luyện (36), trong thực (11), trong thực tế (10)"
mistake,68,"một sai (18), một sai lầm (18), là một sai (13), sai lầm (44), lầm (47)"
chưa vững,0,
nắm chắc,0,
bài kiểm tra,98,"test (55), tests (18), a test (9), exam (7), the test (7)"


TED talks have almost none of the study vocabulary: "chưa vững" and "nắm chắc" never occur, "shaky" occurs in
8 sentences and "solid" mostly means *chất rắn* (a solid). So the corpus cannot choose the interface's words; those
stay hand-written and are guarded by `tests/test_ui_glossary.py`. What the corpus can do is measure a translation
model, which is the rest of this notebook.

## 2. The two models on the IWSLT test set

All 1,268 test pairs. Greedy decoding for both (section 4 shows beams add nothing for envit5).

In [3]:
DEVICE = 'mps' if torch.backends.mps.is_available() else 'cpu'
MODELS = {'opus-mt-en-vi': ('Helsinki-NLP/opus-mt-en-vi', '', ''), 'envit5': ('VietAI/envit5-translation', 'en: ', 'vi: ')}
loaded = {}
def translator(name):
    if name not in loaded:
        repo, prefix, strip = MODELS[name]
        tok = AutoTokenizer.from_pretrained(repo)
        model = AutoModelForSeq2SeqLM.from_pretrained(repo).to(DEVICE).eval()
        def run(sentences, beams=1, batch=16):
            out = []
            for i in range(0, len(sentences), batch):
                b = tok([prefix + s for s in sentences[i:i+batch]], return_tensors='pt', padding=True,
                        truncation=True, max_length=256).to(DEVICE)
                with torch.no_grad():
                    g = model.generate(**b, num_beams=beams, max_new_tokens=256)
                out += [o.removeprefix(strip).strip() for o in tok.batch_decode(g, skip_special_tokens=True)]
            return out
        loaded[name] = run
    return loaded[name]

src, ref = [e for e, _ in test], [v for _, v in test]
scores, outputs = [], {}
for name in MODELS:
    t0 = time.time(); hyp = translator(name)(src); secs = time.time() - t0
    outputs[name] = hyp
    scores.append({'model': name, 'BLEU': round(sacrebleu.corpus_bleu(hyp, [ref]).score, 1),
                   'chrF': round(sacrebleu.corpus_chrf(hyp, [ref]).score, 1), 'sentences/s': round(len(src) / secs, 1)})
pd.DataFrame(scores).style.hide(axis='index')

model,BLEU,chrF,sentences/s
opus-mt-en-vi,30.500000,48.500000,0.200000
envit5,37.400000,55.200000,8.700000


envit5 is ahead on the benchmark. A caveat: it was trained on MTet and PhoMT, multi-domain collections with TED
subtitles among them, so this test set is close to home for it; the app's texts (next section) are the fairer check.

## 3. The app's own texts

The interface shows key point names (2–6 words), key points (one or two sentences), map reasons and overviews,
mostly science. There is no reference translation, so the outputs are shown side by side to be read.

In [4]:
sample = json.loads((config.REPORTS_DIR / 'translation_domain_texts.json').read_text())
def by_text(name, texts):
    pieces = [split_sentences(t) for t in texts]
    flat = translator(name)([s for p in pieces for s in p])
    out, at = [], 0
    for p in pieces:
        out.append(' '.join(flat[at:at+len(p)])); at += len(p)
    return out
texts = [r['en'] for r in sample]
side = pd.DataFrame({'kind': [r['kind'] for r in sample], 'English': texts,
                     'opus-mt-en-vi': by_text('opus-mt-en-vi', texts), 'envit5': by_text('envit5', texts)})
pd.set_option('display.max_colwidth', 160)
side[side.kind == 'label']

,kind,English,opus-mt-en-vi,envit5
0,label,Virus Structure,Nguyên nhân cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu cấu ...,Cấu trúc Vi rútName
1,label,Dalton's law,# do do do,Định luật Dalton
2,label,Polarity and Dissolution,Sự khốn khổ và bất bất bất bất và vô nạn nạn nạn nạn nạn nạn nạn nạn nạn nạn nạn nạn nạn và chia chia chia chia chia chia chia chia chia chia,Phân cực và tan rã
3,label,Cell division for growth,phân tích phần phần phần phần trong phân tích phân l l l còn phân phân tích của tế phân l l cũng còn l l l cũng còn l l l l l l l l l l l cũng gói phần phần...,Phân chia tế bào để tăng trưởng
4,label,Convection Heat Transfer,Commentm tìm tin tin đôi qua đêm qua qua qua quaCommentCommentCommentCommentCommentCommentCommentCommentCommentCommentCommentCommentGenericNameGenericNameGe...,Truyền nhiệt đối lưu
5,label,Glucose for Energy,Lục đơn hướng hướng dẫn y y cho tỷ mực mực mực mực mực mực mực mực cho tỷ đẹpStencil,Glucose để tạo năng lượng
6,label,Overpopulation and Extinction,Vô vô và muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn muôn mu...,Bùng nổ dân số và tuyệt chủng
7,label,FM Radio Noise Rejection,Comment,Từ chối tiếng ồn radio FM
8,label,Ionic Bonding,Sự kết kết nối giữa kết nối,Liên kết Ion
9,label,Genetics and DNA inheritance,♪ và DNA và ♪,Di truyền học và di truyền DNA


In [5]:
side[side.kind != 'label']

,kind,English,opus-mt-en-vi,envit5
12,point,Mass wasting is a form of erosion where gravity pulls soil and rocks directly down slopes.,Sự tiêu hao hàng loạt là một dạng của đường kính mà trọng lực hút đất và đá trực tiếp xuống các dốc.,Sạt lở là một hình thức xói mòn khi trọng lực kéo đất và đá trực tiếp xuống sườn dốc.
13,point,"The Sun is Earth's primary energy source, producing immense energy through nuclear fusion of hydrogen into helium. This energy reaches Earth primarily as vi...","Mặt trời là nguồn năng lượng chính của Trái Đất, sản xuất năng lượng khổng lồ thông qua sự tổng hợp của năng lượng hạt nhân vào trong khí quyển. Năng lượng ...","Mặt trời là nguồn năng lượng chính của Trái đất, tạo ra năng lượng khổng lồ thông qua phản ứng tổng hợp hạt nhân của hydro thành heli. Năng lượng này đến Tr..."
14,point,"Air is a poor conductor of heat and acts as a good insulator, especially when trapped in small spaces that prevent convection. This principle is used in wal...","Phong là một bộ điều hòa nhiệt thấp và hoạt động như một chất cách nhiệt tốt, đặc biệt khi bị kẹt trong những khoảng không nhỏ ngăn cản sự va chạm. Nguyên l...","Không khí là một chất dẫn nhiệt kém và hoạt động như một chất cách điện tốt, đặc biệt là khi bị mắc kẹt trong không gian nhỏ ngăn cản sự đối lưu. Nguyên lý ..."
15,point,"Biological systems contain macromolecules, which are extremely large molecules composed of many atoms that can fold back on themselves, leading to intramole...","Các hệ sinh học có cấu trúc mô bào, là những phân tử cực kỳ lớn gồm nhiều nguyên tử có thể tự cuộn lại, dẫn đến sự tương tác trong màng não.","Các hệ thống sinh học chứa các đại phân tử, là các phân tử cực kỳ lớn bao gồm nhiều nguyên tử có thể tự gấp lại, dẫn đến tương tác nội phân tử."
16,point,"Carbon forms various carbides, including ionic with electropositive metals, interstitial with transition metals, and covalent with elements of similar elect...","Các dạng các đường nối các đường nối khác nhau, bao gồm iron với kim loại điện cực, các dây liên kết với các kim loại liên tiếp, và liên kết với các thành p...","Cacbon tạo thành nhiều cacbua khác nhau, bao gồm ion với kim loại điện dương, kẽ hở với kim loại chuyển tiếp, và cộng hoá trị với các nguyên tố có độ âm điệ..."
17,point,"Voltage represents energy per unit charge, not the total energy stored. Two batteries can have the same voltage but different energy storage capacities if o...","Kích thước tạo năng lượng cho mỗi đơn vị nạp, không phải tổng năng lượng. Hai pin có thể có cùng một khối nhưng các kho năng lượng khác nhau nếu một người c...","Điện áp đại diện cho năng lượng trên mỗi đơn vị điện tích, không phải là tổng năng lượng được lưu trữ. Hai pin có thể có cùng điện áp nhưng dung lượng lưu t..."
18,point,Enzymes speed up the biochemical reactions in living organisms so cellular processes happen quickly enough to sustain life.,Sự chậm dần tăng tốc độ hóa sinh trong cơ thể sống để các tế bào có thể nhanh chóng tồn tại sự sống.,Enzyme đẩy nhanh các phản ứng sinh hoá trong sinh vật sống để quá trình tế bào diễn ra đủ nhanh để duy trì sự sống.
19,point,"Ecological succession describes the gradual process where one community of species is replaced by others over time, starting with pioneer species and progre...","Các nhà thần học mô tả quá trình mà một cộng đồng các loài được thay thế bởi những loài khác theo thời gian, bắt đầu với loài tiên phong và trải qua những g...","Kế thừa sinh thái mô tả quá trình dần dần trong đó một cộng đồng loài được thay thế bởi những loài khác theo thời gian, bắt đầu với các loài tiên phong và t..."
20,point,"In metallic solids, valence electrons are delocalized throughout the crystal, creating a strong cohesive force that gives metals properties like luster, hig...","Trong các khối rắn, các điện tử được tạo ra từ khắp trong tinh thể, tạo ra một lực liên kết mạnh mẽ tạo ra các hạt kim loại như những hạt xốp, sự dẫn dắt ca...","Trong chất rắn kim loại, các electron hoá trị được phân địn

In [6]:
# A name the model loops on repeats one syllable many times.
looping = lambda s: max(Counter(s.split()).values(), default=0) >= 4 and len(s.split()) > 8
print({name: int(side[name].map(looping).sum()) for name in ('opus-mt-en-vi', 'envit5')}, 'of', len(side), 'texts loop')

{'opus-mt-en-vi': 11, 'envit5': 2} of 30 texts loop


Reading the two columns: opus-mt-en-vi loops on short names ("Nguyên cấu cấu cấu …", "CommentComment…") and
changes the meaning of whole sentences ("Energy conservation" became *Thuyết tiến hóa*, evolution). envit5 gets
nearly all of them right. Its mistakes are literal readings of terms: *Từ chối tiếng ồn* for "noise rejection"
(should be *chống nhiễu*), *Kế thừa sinh thái* for "ecological succession" (*diễn thế sinh thái*), *cách điện*
(electrical) for an air *insulator* of heat (*cách nhiệt*). That is why the page keeps the document's own words one
hover away, and keeps the quoted passage under each key point untranslated.

## 4. Speed on this machine, and the choice

In [7]:
sents = [s for t in texts for s in split_sentences(t)]
rows = []
for beams in (1, 4):
    t0 = time.time(); hyp = translator('envit5')(sents, beams=beams); secs = time.time() - t0
    sub = src[:200]
    bleu = sacrebleu.corpus_bleu(translator('envit5')(sub, beams=beams), [ref[:200]]).score
    rows.append({'beams': beams, 'app sentences/s': round(len(sents) / secs, 1), 'BLEU (first 200 test)': round(bleu, 1)})
pd.DataFrame(rows).style.hide(axis='index')

beams,app sentences/s,BLEU (first 200 test)
1,11.700000,36.300000
4,3.200000,36.700000


**Choice: `VietAI/envit5-translation`, greedy, on the Apple GPU when there is one** (`ml_pipeline/translator.py`).
Beams cost time and add nothing measurable here. One model covers both directions, so a Vietnamese document on an
English page works the same way. A document's key points (about 45 sentences) take a few seconds the first time;
each text is kept after it is translated once (`notebooks.Translation`), so every later view is instant, and the
page shows the document's own words until the translation arrives.

**Not done.** Fine-tuning on the IWSLT training pairs: they are TED subtitles, not study material, and envit5 has
already seen TED subtitles. A hand-labelled set of study texts with reference translations would be the next measurement.

## 5. Two faults on the app's texts, and what the translator does about them

Reading the demo account's 89 key point names showed two faults the benchmark cannot see.

**Names in Title Case.** The model has seen Title Case mostly in software menus, so "Virus Structure" came back as
*Cấu trúc Vi rútName* and "Nutrient Cycles" as *Công thức tính toán* (calculation formula). The translator now
puts such a name in sentence case first (acronyms and mixed-case words like DNA, mRNA, Brønsted-Lowry stay as
written) and gives the result a capital first letter. The names whose translation changed:

In [8]:
from ml_pipeline import translator as local
names = json.loads((config.REPORTS_DIR / 'translation_demo_labels.json').read_text())['labels']
before = translator('envit5')(names)
after = local.translate(names, 'en', 'vi')
changed = pd.DataFrame({'name': names, 'as written': before, 'in sentence case': after})
changed = changed[changed['as written'] != changed['in sentence case']]
print(f'{len(changed)} of {len(names)} names translate differently')
changed.style.hide(axis='index')

31 of 92 names translate differently


name,as written,in sentence case
Acid Strength,Độ bền của axit,Độ mạnh của axit
Air's Insulating Properties,Thuộc tính cách điện của không khí,Đặc tính cách nhiệt của không khí
Atomic Mass and Isotopes,Khối lượng nguyên tử và đồng vị,Nguyên tử khối và đồng vị
Bond Energy and Stability,Năng lượng liên kết và tính ổn định,Năng lượng liên kết và sự ổn định
Brønsted-Lowry Acids and Bases,Axit và bazơ Brnsted-Lowry,Brønsted-Lowry axit và bazơ
Competition and Adaptation,Cạnh tranh và Thích nghi,Cạnh tranh và thích nghi
Covalent Bonding,Liên kết cộng hóa trị,Liên kết cộng hoá trị
Diverse Carbides,Carb đa dạng,Các cacbua đa dạng
EM Wave Energy and Frequency,Sóng EM Năng lượng và tần số,Năng lượng và tần số sóng EM
Ecological Succession,Kế vị sinh thái,Kế thừa sinh thái


Read side by side, most of the changed names read better in sentence case (*Chu trình dinh dưỡng*, *Chọn lọc tự
nhiên*, *cách nhiệt* for an insulator of heat, *Hình dạng phân tử quyết định tính chất*), a few read the same, and
two read worse ("Phase Change Energy", "Ring Species"). The trade is clearly worth it.

**Symbols.** The model's vocabulary has no λ Δ ≈ ≤ → ° ø, so it drops them without a sign, and it reads super-
and subscripts as plain digits (m² became m2, Cl⁻ became Cl). In a science text that changes the meaning
("pH ≤ 7" became "pH 7"). The translator now sends each run of such characters as a placeholder `<0>`, `<1>`, …,
which the model copies through, and puts the characters back. A word holding such a letter goes whole
("Brønsted-Lowry"): with only the ø replaced, the model garbled the rest of the name. If a placeholder is lost the sentence is left as
written rather than shown with a symbol missing.

In [9]:
symbols = ['Water (H₂O) boils at 100 °C at 1 atm.', 'The wavelength λ equals v ÷ f, so Δv ≈ 0 means no change.',
           'CO₂ + H₂O → C₆H₁₂O₆ + O₂ in photosynthesis.', 'Acids have pH ≤ 7.', 'Brønsted-Lowry acids donate protons.',
           'Ions such as Na⁺ and Cl⁻ conduct electricity in water.', 'The area is 4 m² and the angle is 30°.']
pd.DataFrame({'English': symbols, 'model alone': translator('envit5')(symbols),
              'translator': local.translate(symbols, 'en', 'vi')}).style.hide(axis='index')

English,model alone,translator
Water (H₂O) boils at 100 °C at 1 atm.,Nước (H 2 O) sôi ở 100 C tại 1 atm.,Nước (H₂O) sôi ở 100 °C tại 1 atm.
"The wavelength λ equals v ÷ f, so Δv ≈ 0 means no change.","Bước sóng bằng v f, vì vậy v 0 có nghĩa là không thay đổi.","Bước sóng λ bằng v ÷ f, vì vậy Δv ≈ 0 có nghĩa là không thay đổi."
CO₂ + H₂O → C₆H₁₂O₆ + O₂ in photosynthesis.,CO2 + H2O C6H12O6 + O2 trong quang hợp.,CO₂ + H₂O → C₆H₁₂O₆ + O₂ trong quang hợp.
Acids have pH ≤ 7.,Axit có pH 7.,Axit có pH ≤ 7.
Brønsted-Lowry acids donate protons.,Axit Brnsted-Lowry cho proton.,Brønsted-Lowry axit tặng proton.
Ions such as Na⁺ and Cl⁻ conduct electricity in water.,Các ion như Na+ và Cl dẫn điện trong nước.,Các ion như Na⁺ và Cl⁻ dẫn điện trong nước.
The area is 4 m² and the angle is 30°.,Diện tích là 4 m2 và góc là 30.,Diện tích là 4 m² và góc là 30°.


Neither change should cost anything on ordinary sentences. The whole IWSLT test set again, through the translator:

In [10]:
raw = sacrebleu.corpus_bleu(translator('envit5')(src), [ref])
full = sacrebleu.corpus_bleu(local.translate(src, 'en', 'vi'), [ref])
pd.DataFrame([{'': 'model alone', 'BLEU': round(raw.score, 1)}, {'': 'translator', 'BLEU': round(full.score, 1)}]).style.hide(axis='index')

,BLEU
model alone,37.400000
translator,37.300000


The cache key includes `translator.VERSION`, so names kept from before these fixes are translated again.

**Still wrong, left to the hover.** Some science terms are read literally: *Kế thừa sinh thái* for ecological
succession (*diễn thế sinh thái*), *Từ chối nhiễu* for noise rejection (*chống nhiễu*), *Áp lực nội nhãn* for
intrapleural pressure. Writing the Vietnamese term into the English sentence before translating was tried: the
model kept it in short names but garbled or dropped it in half of the sentences, so it is not used.

## 6. Flashcards and quizzes: short answers

A flashcard's answer or a quiz choice is often one or two words, and a word alone has no sense to translate:
"Force" came back as *Buộc* (to force), "nguyên phân" as *integral*. Wrapping the word ("The term …", "In science,
…") did not help. What helps is the question: the answer is sent as `<question> Answer: <answer>.` and read back
after the answer word the model writes (*Đáp án:*, *Câu trả lời:* …); if it writes none, the word goes alone.

In [11]:
pairs = [("What is required to change an object's angular velocity in rotational motion?", 'Force'),
         ('The process in which the nucleus divides is called what?', 'cytokinesis'),
         ('The process in which the nucleus divides is called what?', 'mitosis'),
         ('The process in which the nucleus divides is called what?', 'epistasis'),
         ('What quantity is measured in kilograms?', 'Mass'),
         ('What is transferred when a force moves an object?', 'Work'),
         ('What property makes a particle feel an electric force?', 'Charge'),
         ('What is the rate of doing work called?', 'Power')]
pd.DataFrame({'question': [q for q, _ in pairs], 'answer': [a for _, a in pairs],
              'alone': local.translate([a for _, a in pairs], 'en', 'vi'),
              'with its question': local.translate_in_context([(a, q) for q, a in pairs], 'en', 'vi')}).style.hide(axis='index')

question,answer,alone,with its question
What is required to change an object's angular velocity in rotational motion?,Force,Buộc,Lực
The process in which the nucleus divides is called what?,cytokinesis,tế bào học,sự phân chia tế bào
The process in which the nucleus divides is called what?,mitosis,nguyên phân,nguyên phân
The process in which the nucleus divides is called what?,epistasis,epistasis,sự lan truyền
What quantity is measured in kilograms?,Mass,Khối,Khối lượng
What is transferred when a force moves an object?,Work,Công trình,Công việc
What property makes a particle feel an electric force?,Charge,Nạp điện,Điện tích
What is the rate of doing work called?,Power,Quyền lực,Quyền lực


With its question the model gets *Lực*, *khối lượng*, *Điện tích* and *nguyên phân* right where alone it got
*Buộc*, *Khối*, *Nạp điện*. It is still wrong on some physics words (*Công việc* for work, *Quyền lực* for power), so
the page shows a translated short answer with the document's word beside it rather than alone. The same in the
other direction is how an answer typed in Vietnamese to an English question is graded, with the
question first put into Vietnamese (a prompt in two languages confuses the model):

In [12]:
typed = [('nguyên phân', 'The process in which the nucleus divides is called what?'),
         ('khối lượng', 'What quantity is measured in kilograms?'),
         ('động năng', 'Temperature is a measure of the average amount of energy of motion, or ___, a system contains.')]
# The question goes in the answer's language first: a prompt in two languages confuses the model.
asked = list(zip([t for t, _ in typed], local.translate([q for _, q in typed], 'en', 'vi')))
pd.DataFrame({'typed': [t for t, _ in typed], 'alone': local.translate([t for t, _ in typed], 'vi', 'en'),
              'with its question': local.translate_in_context(asked, 'vi', 'en')}).style.hide(axis='index')

typed,alone,with its question
nguyên phân,integral,mitosis
khối lượng,volume,mass
động năng,kinetic energy,kinetic energy


A fill-in-the-blank card keeps its blank: `___` travels as a placeholder like the symbols in section 5.

In [13]:
local.translate(['Temperature is a measure of the average amount of energy of motion, or ___, a system contains.'], 'en', 'vi')

['Nhiệt độ là thước đo lượng năng lượng trung bình của chuyển động, hoặc ___, một hệ thống chứa.']